# Clean IVV wine production data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oxedanda/pml_final_project/blob/main/notebooks/ivv_wine_production.ipynb)

This notebook cleans the IVV file `36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xls`.

It clones the GitHub repository, reads the raw Excel file from `data/raw`, transforms the tables into a machine-learning-ready format, and saves the processed CSV in:

```text
/content/pml_final_project/data/processed/wine_production_by_region_clean.csv
```


In [ ]:
!pip install -q pandas numpy scikit-learn xlrd openpyxl lxml

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REPO_URL = "https://github.com/oxedanda/pml_final_project.git"
ROOT_DIR = Path("/content/pml_final_project")

if not ROOT_DIR.exists():
    !git clone {REPO_URL} /content/pml_final_project
else:
    print("Repository already exists. Pulling latest changes...")
    %cd /content/pml_final_project
    !git pull

%cd /content/pml_final_project

RAW_FILE = ROOT_DIR / "data" / "raw" / "36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xls"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
OUTPUT_FILE = PROCESSED_DIR / "wine_production_by_region_clean.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Root directory:", ROOT_DIR)
print("Raw file:", RAW_FILE)
print("Output file:", OUTPUT_FILE)
print("Raw file exists?", RAW_FILE.exists())


In [ ]:
def load_excel_or_html_tables(file_path):
    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            "Check whether the file exists in data/raw in the GitHub repository."
        )

    try:
        sheets = pd.read_excel(file_path, sheet_name=None, header=None, dtype=str)
        print(f"Loaded {len(sheets)} Excel sheet(s).")
        return sheets
    except Exception as excel_error:
        print("pd.read_excel failed. Trying pd.read_html...")
        print("Excel error:", excel_error)

        html_tables = pd.read_html(file_path, header=None)
        sheets = {f"html_table_{i+1}": table.astype(str) for i, table in enumerate(html_tables)}
        print(f"Loaded {len(sheets)} HTML table(s).")
        return sheets


raw_sheets = load_excel_or_html_tables(RAW_FILE)

for sheet_name, df in raw_sheets.items():
    print(f"{sheet_name}: {df.shape}")


In [ ]:
CAMPAIGN_PATTERN = re.compile(r"^20\d{2}/\d{2}$")

TYPE_MAP = {
    "total": "total_production_hl",
    "dop": "dop_production_hl",
    "igp": "igp_production_hl",
    "ano_casta": "year_variety_production_hl",
    "sem_do_ig": "non_certified_production_hl",
}

REGIONS_TO_KEEP = {
    "Verdes",
    "T. Montes",
    "Douro",
    "Bairrada",
    "Dão",
    "Beira Interior",
    "Távora-Varosa",
    "Tejo",
    "Lisboa",
    "P. Setúbal",
    "Alentejo",
    "Algarve",
    "Madeira",
    "Açores",
}


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).replace("\xa0", " ").strip()
    value = re.sub(r"\s+", " ", value)
    return value


def clean_number_pt(value):
    text = normalize_text(value)

    if text == "" or text.lower() in {"nan", "none"}:
        return np.nan

    text = re.sub(r"[^0-9,\.\-]", "", text)

    if text == "":
        return np.nan

    text = text.replace(".", "").replace(",", ".")

    try:
        return float(text)
    except ValueError:
        return np.nan


def infer_table_type(title_text):
    title = normalize_text(title_text).lower()

    if "total por região" in title or "total por regiao" in title:
        return "total"

    if "denominação de origem protegida" in title or "denominacao de origem protegida" in title or "dop" in title:
        return "dop"

    if "indicação geográfica protegida" in title or "indicacao geografica protegida" in title or "igp" in title:
        return "igp"

    if "ano/casta" in title or "ano / casta" in title:
        return "ano_casta"

    if "sem do/ig" in title or "sem do / ig" in title:
        return "sem_do_ig"

    return None


def find_title_above(df, header_row, max_rows_above=8):
    start = max(0, header_row - max_rows_above)
    candidate_rows = []

    for r in range(start, header_row):
        row_text = " ".join(normalize_text(x) for x in df.iloc[r].tolist())
        if "Evolução" in row_text or "Evolucao" in row_text:
            candidate_rows.append(row_text)

    if candidate_rows:
        return candidate_rows[-1]

    return ""


def detect_header_rows(df):
    header_rows = []

    for idx in range(len(df)):
        row_values = [normalize_text(x) for x in df.iloc[idx].tolist()]
        campaign_count = sum(bool(CAMPAIGN_PATTERN.match(x)) for x in row_values)

        if campaign_count >= 3:
            header_rows.append(idx)

    return header_rows


def parse_table_from_header(df, sheet_name, header_row):
    title = find_title_above(df, header_row)
    table_type = infer_table_type(title)

    if table_type is None:
        print(f"Skipping table in sheet '{sheet_name}', row {header_row}: could not infer table type.")
        print("Title found:", title[:120])
        return None

    header_values = [normalize_text(x) for x in df.iloc[header_row].tolist()]

    year_columns = {
        col_idx: value
        for col_idx, value in enumerate(header_values)
        if CAMPAIGN_PATTERN.match(value)
    }

    if not year_columns:
        return None

    first_year_col = min(year_columns.keys())
    region_col = max(0, first_year_col - 1)

    records = []

    for r in range(header_row + 1, len(df)):
        region = normalize_text(df.iat[r, region_col])

        if region.lower() in {"total geral", "total"}:
            break

        if region == "" or region.lower() in {"nan", "região vitivinícola", "regiao vitivinicola"}:
            continue

        if region not in REGIONS_TO_KEEP:
            continue

        for c, campaign_year in year_columns.items():
            value = clean_number_pt(df.iat[r, c])

            records.append({
                "sheet_name": sheet_name,
                "production_type": table_type,
                "production_column": TYPE_MAP[table_type],
                "region": region,
                "campaign_year": campaign_year,
                "year_start": int(campaign_year[:4]),
                "production_hl": value,
            })

    if not records:
        return None

    return pd.DataFrame(records)


In [ ]:
all_long_tables = []

for sheet_name, df in raw_sheets.items():
    header_rows = detect_header_rows(df)
    print(f"Sheet/table '{sheet_name}' - detected header rows: {header_rows}")

    for header_row in header_rows:
        parsed = parse_table_from_header(df, sheet_name, header_row)

        if parsed is not None:
            all_long_tables.append(parsed)
            table_type = parsed["production_type"].iloc[0]
            print(f"  Parsed {table_type}: {parsed.shape[0]} rows")

if not all_long_tables:
    raise ValueError(
        "No valid production tables were parsed. "
        "Check if the header rows contain campaign years such as 2025/26."
    )

production_long = pd.concat(all_long_tables, ignore_index=True)

display(production_long.head())
print(production_long.shape)

summary = (
    production_long
    .groupby("production_type")
    .agg(
        rows=("production_hl", "size"),
        regions=("region", "nunique"),
        campaigns=("campaign_year", "nunique"),
        missing_values=("production_hl", lambda x: x.isna().sum()),
    )
    .reset_index()
)

display(summary)


In [ ]:
production_wide = (
    production_long
    .pivot_table(
        index=["region", "campaign_year", "year_start"],
        columns="production_column",
        values="production_hl",
        aggfunc="first",
    )
    .reset_index()
)

production_wide.columns.name = None

expected_value_columns = list(TYPE_MAP.values())

for col in expected_value_columns:
    if col not in production_wide.columns:
        production_wide[col] = np.nan

production_wide = production_wide[
    ["region", "campaign_year", "year_start"] + expected_value_columns
]

production_wide = production_wide.sort_values(["year_start", "region"]).reset_index(drop=True)

display(production_wide.head(20))
print(production_wide.shape)

production_wide.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("Saved processed dataset to:")
print(OUTPUT_FILE)


In [ ]:
check = pd.read_csv(OUTPUT_FILE)

display(check.head(20))
print("Shape:", check.shape)

print("\nColumns:")
print(check.columns.tolist())

print("\nYears:")
print(sorted(check["campaign_year"].unique()))

print("\nRegions:")
print(sorted(check["region"].unique()))

print("\nDuplicated region-year rows:")
print(check.duplicated(subset=["region", "campaign_year"]).sum())

print("\nMissing values:")
display(check.isna().sum().to_frame("missing_count"))

print("\nTarget summary:")
display(check["total_production_hl"].describe())


In [ ]:
from google.colab import files

files.download(str(OUTPUT_FILE))
